# MotionForge 0.3 — Character + Object Motion Capture
This notebook extracts experimental human BVH motion and can optionally track one selected object (ball/box/bottle) from a manually selected first-frame bounding box. Object export is approximate 2D screen-plane motion, not full 3D reconstruction or reliable grasp detection.

In [ ]:
!pip -q install uv
!uv python install 3.12
!uv venv --clear --python 3.12 /content/motionforge-venv\n!uv pip install --reinstall --python /content/motionforge-venv/bin/python "git+https://github.com/sohailkhanbuilds/MotionForge.git"


In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = next(iter(uploaded))
print('Input video:', video_path)

## Optional: select an object to track
Inspect the first frame, then enter the object's bounding box in pixels: x, y, width, height. For body-only motion, skip these cells and run the normal processing command.

In [ ]:
track_object = input('Track an object too? (y/n): ').strip().lower() == 'y'
if track_object:
    import cv2
    from google.colab.patches import cv2_imshow
    cap = cv2.VideoCapture(video_path)
    ok, first_frame = cap.read()
    cap.release()
    if not ok: raise ValueError('Could not read the first video frame')
    display_width = min(900, first_frame.shape[1])
    display_height = int(first_frame.shape[0] * display_width / first_frame.shape[1])
    print('Original image size:', first_frame.shape[1], 'x', first_frame.shape[0])
    cv2_imshow(cv2.resize(first_frame, (display_width, display_height)))
    scale = first_frame.shape[1] / display_width
    roi_x = int(float(input('Object ROI x on displayed image: ')) * scale)
    roi_y = int(float(input('Object ROI y on displayed image: ')) * scale)
    roi_w = int(float(input('Object ROI width on displayed image: ')) * scale)
    roi_h = int(float(input('Object ROI height on displayed image: ')) * scale)
    object_type = input('Object type (ball/box/bottle/cylinder/cone): ').strip().lower() or 'ball'


In [ ]:
import subprocess
cmd = ['/content/motionforge-venv/bin/python', '-m', 'motionforge', '--input', video_path, '--output-dir', '/content/motionforge_output', '--model-complexity', '0', '--smooth-window', '5']
if track_object:
    cmd += ['--object-roi', str(roi_x), str(roi_y), str(roi_w), str(roi_h), '--object-type', object_type]
subprocess.run(cmd, check=True, env={**__import__('os').environ, 'MPLBACKEND': 'Agg'})


In [ ]:
from google.colab import files
from pathlib import Path
for filename in ['motionforge.bvh', 'landmarks.csv', 'pose_preview.mp4', 'metadata.json']:
    files.download('/content/motionforge_output/' + filename)
for filename in ['object_motion.json', 'object_animation.py']:
    path = Path('/content/motionforge_output') / filename
    if path.exists(): files.download(str(path))
